In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the clinical data
clinical = pd.read_csv('luad_tcga_pan_can_atlas_2018_clinical_data.tsv', sep='\t')
clinical.head()
clinical.columns.tolist()

['Study ID',
 'Patient ID',
 'Sample ID',
 'Diagnosis Age',
 'Neoplasm Disease Stage American Joint Committee on Cancer Code',
 'American Joint Committee on Cancer Publication Version Type',
 'Aneuploidy Score',
 'Buffa Hypoxia Score',
 'Cancer Type',
 'TCGA PanCanAtlas Cancer Type Acronym',
 'Cancer Type Detailed',
 'Last Communication Contact from Initial Pathologic Diagnosis Date',
 'Birth from Initial Pathologic Diagnosis Date',
 'Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value',
 'Disease Free (Months)',
 'Disease Free Status',
 'Months of disease-specific survival',
 'Disease-specific Survival status',
 'Ethnicity Category',
 'Form completion date',
 'Fraction Genome Altered',
 'Genetic Ancestry Label',
 'Neoplasm Histologic Grade',
 'Neoadjuvant Therapy Type Administered Prior To Resection Text',
 'ICD-10 Classification',
 'International Classification of Diseases for Oncology, Third Edition ICD-O-3 Histology Code',
 'International Classification of Diseas

In [ ]:
#Create a new dataframe with only the relevant columns for our analysis
clinical_clean = clinical[['Patient ID', 'Overall Survival (Months)', 'Overall Survival Status',
                            'Diagnosis Age', 'Sex', 
                            'Neoplasm Disease Stage American Joint Committee on Cancer Code']].copy()

In [ ]:
# Rename columns for easier access
clinical_clean.columns = ["patient_id",
    "overall_survival_months",
    "overall_survival_status",
    "age",
    "gender",
    "stage"]

clinical_clean.head(10)

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage
0,TCGA-05-4244,0.000000,0:LIVING,70.0,Male,STAGE IV
1,TCGA-05-4249,50.070684,0:LIVING,67.0,Male,STAGE IB
2,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,STAGE IIIA
3,TCGA-05-4382,19.955946,0:LIVING,68.0,Male,STAGE IB
4,TCGA-05-4384,14.005326,0:LIVING,66.0,Male,STAGE IIIA
5,TCGA-05-4389,45.007726,0:LIVING,70.0,Male,STAGE IA
6,TCGA-05-4390,37.018772,0:LIVING,58.0,Female,STAGE IB
7,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,STAGE IIIB
8,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,STAGE IIIB
9,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,STAGE IIB


In [ ]:
# Create an 'event' column where 1 indicates deceased and 0 indicates alive
clinical_clean['event'] = clinical_clean["overall_survival_status"].str.contains("DECEASED").astype(int)

In [ ]:
clinical_clean.head(10)

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event
0,TCGA-05-4244,0.000000,0:LIVING,70.0,Male,STAGE IV,0
1,TCGA-05-4249,50.070684,0:LIVING,67.0,Male,STAGE IB,0
2,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,STAGE IIIA,1
3,TCGA-05-4382,19.955946,0:LIVING,68.0,Male,STAGE IB,0
4,TCGA-05-4384,14.005326,0:LIVING,66.0,Male,STAGE IIIA,0
5,TCGA-05-4389,45.007726,0:LIVING,70.0,Male,STAGE IA,0
6,TCGA-05-4390,37.018772,0:LIVING,58.0,Female,STAGE IB,0
7,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,STAGE IIIB,1
8,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,STAGE IIIB,1
9,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,STAGE IIB,1


In [ ]:
# Create a 'survival_5yr' column where 1 indicates survival of at least 5 years and 0 indicates death within 5 years
clinical_clean["survival_5yr"] = np.where(
    (clinical_clean["overall_survival_months"] >= 60) &
    (clinical_clean["event"] == 0),
    1,
    np.where(
        (clinical_clean["overall_survival_months"] < 60) &
        (clinical_clean["event"] == 1),
        0,
        np.nan
    )
)

In [ ]:
# Drop rows with missing values in the 'survival_5yr' column
clinical_clean = clinical_clean.dropna(subset=["survival_5yr"])

In [ ]:
# Check the distribution of the 'survival_5yr' column
clinical_clean["survival_5yr"].value_counts()

survival_5yr
0.0    171
1.0     42
Name: count, dtype: int64

In [ ]:
# Check for missing values in the cleaned clinical data
clinical_clean.isna().sum()

patient_id                 0
overall_survival_months    0
overall_survival_status    0
age                        8
gender                     0
stage                      0
event                      0
survival_5yr               0
dtype: int64

In [ ]:
clinical_clean.head(20)

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event,survival_5yr
2,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,STAGE IIIA,1,0.0
7,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,STAGE IIIB,1,0.0
8,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,STAGE IIIB,1,0.0
9,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,STAGE IIB,1,0.0
11,TCGA-05-4402,8.021830,1:DECEASED,57.0,Female,STAGE IV,1,0.0
15,TCGA-05-4415,2.991748,1:DECEASED,57.0,Male,STAGE IIIB,1,0.0
17,TCGA-05-4418,9.008120,1:DECEASED,69.0,Male,STAGE IIIA,1,0.0
27,TCGA-05-4434,15.024493,1:DECEASED,67.0,Female,STAGE IV,1,0.0
31,TCGA-05-5429,9.040997,1:DECEASED,60.0,Male,STAGE IIIA,1,0.0
89,TCGA-38-4625,97.741395,0:LIVING,66.0,Female,STAGE IB,0,1.0


In [ ]:
# Extract the stage information using a regular expression to capture the stage in the format of I, II, III, IV
clinical_clean["stage"] = (
    clinical_clean["stage"]
    .astype(str)
    .str.extract(r"(I{1,3}V?|IV)")
)

In [ ]:
# Add the "Stage " prefix to the stage values
clinical_clean["stage"] = "Stage " + clinical_clean["stage"]

In [ ]:
# Fill missing stage values with "Unknown"
clinical_clean["stage"] = clinical_clean["stage"].fillna("Unknown")

In [ ]:
clinical_clean["stage"].value_counts()

stage
Stage I      94
Stage II     56
Stage III    48
Stage IV     15
Name: count, dtype: int64

In [ ]:
clinical_clean.head(10)

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event,survival_5yr
2,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,Stage III,1,0.0
7,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,Stage III,1,0.0
8,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,Stage III,1,0.0
9,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,Stage II,1,0.0
11,TCGA-05-4402,8.021830,1:DECEASED,57.0,Female,Stage IV,1,0.0
15,TCGA-05-4415,2.991748,1:DECEASED,57.0,Male,Stage III,1,0.0
17,TCGA-05-4418,9.008120,1:DECEASED,69.0,Male,Stage III,1,0.0
27,TCGA-05-4434,15.024493,1:DECEASED,67.0,Female,Stage IV,1,0.0
31,TCGA-05-5429,9.040997,1:DECEASED,60.0,Male,Stage III,1,0.0
89,TCGA-38-4625,97.741395,0:LIVING,66.0,Female,Stage I,0,1.0


In [ ]:
# Save the cleaned clinical data to a new CSV file
clinical_clean.to_csv(
    "LUAD_cleaned_data.csv",
    index=False
)


##### Filter the gene expression file to your 30 LUAD genes and transpose

In [ ]:
# Load the gene expression data
gene_exp = pd.read_csv(
    "data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt",
    sep="\t"
)

gene_exp.head()

,Hugo_Symbol,Entrez_Gene_Id,TCGA-05-4244-01,TCGA-05-4249-01,TCGA-05-4250-01,TCGA-05-4382-01,TCGA-05-4384-01,TCGA-05-4389-01,TCGA-05-4390-01,TCGA-05-4395-01,...,TCGA-NJ-A4YG-01,TCGA-NJ-A4YI-01,TCGA-NJ-A4YP-01,TCGA-NJ-A4YQ-01,TCGA-NJ-A55A-01,TCGA-NJ-A55O-01,TCGA-NJ-A55R-01,TCGA-NJ-A7XG-01,TCGA-O1-A52J-01,TCGA-S2-AA1A-01
0,NaN,100130426,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,...,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883,-2.2883
1,NaN,100133144,0.0422,-0.3456,-0.3377,0.1909,-1.2157,-0.9946,-0.3597,-1.1518,...,0.2311,-1.0855,-1.5285,0.6372,0.9119,0.5749,-0.1628,1.8611,0.4557,0.9231
2,UBE2Q2P2,100134869,0.0641,0.1921,-0.7281,-0.4447,-1.3592,0.3267,-0.4435,0.1590,...,0.1399,-0.1230,-0.3234,-0.4515,0.6240,1.0118,-0.0510,2.7540,1.0811,-0.2340
3,HMGB1P1,10357,-1.9056,-0.2903,-1.9090,-0.5293,-0.8865,-1.1254,0.2158,-1.5307,...,1.3096,0.7032,0.0252,-0.5426,-0.0100,-0.0163,-0.8067,-0.4479,-1.3456,-0.7284
4,NaN,10431,-0.0365,0.1977,0.7798,-0.1759,-1.1759,1.2304,0.7983,0.2914,...,0.1831,-2.0906,0.1825,0.4039,-1.3028,0.0438,-0.3179,-0.6588,0.7717,-1.1591


In [ ]:
gene_exp.shape

(20531, 512)

In [ ]:
gene_exp.columns[:10]

Index(['Hugo_Symbol', 'Entrez_Gene_Id', 'TCGA-05-4244-01', 'TCGA-05-4249-01',
       'TCGA-05-4250-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01',
       'TCGA-05-4389-01', 'TCGA-05-4390-01', 'TCGA-05-4395-01'],
      dtype='str')

In [ ]:
# Extract the target gene names from the first column
selected_genes = [
"EGFR","KRAS","ALK","BRAF","ERBB2","MET","ROS1","RET","PIK3CA","NRAS",
"TP53","STK11","KEAP1","NF1","RB1","SMARCA4","ATM","CDKN2A",
"PDCD1","CD274","CTLA4","CXCL9","CXCL10",
"MKI67","MYC","CCND1","VEGFA","AKT1","MAPK1","MTOR"
]

In [ ]:
# Filter the gene expression data to include only the selected genes
gene_filtered = gene_exp[
    gene_exp["Hugo_Symbol"].isin(selected_genes)
]

In [ ]:
# Drop the 'Entrez_Gene_Id' column as it is not needed for our analysis
gene_filtered = gene_filtered.drop(columns=["Entrez_Gene_Id"])

In [ ]:
# Set the 'Hugo_Symbol' column as the index for easier access to gene expression values
gene_filtered = gene_filtered.set_index("Hugo_Symbol")

In [ ]:
# Transpose the gene expression data so that genes are columns and samples are rows
gene_transposed = gene_filtered.T

In [ ]:
gene_transposed.head()

Hugo_Symbol,AKT1,ALK,ATM,BRAF,CCND1,CD274,CDKN2A,CTLA4,CXCL10,CXCL9,...,NRAS,PDCD1,PIK3CA,RB1,RET,ROS1,SMARCA4,STK11,TP53,VEGFA
TCGA-05-4244-01,-1.4348,0.1593,-1.8591,0.3711,1.2510,1.2876,-0.7482,-0.0031,-0.4304,-0.8660,...,0.5259,-1.0942,-0.2992,-0.4565,-1.1858,0.8186,-0.1573,0.4716,0.1371,2.1437
TCGA-05-4249-01,0.8508,0.2142,0.5106,-0.5496,-0.4023,0.0360,0.3018,-0.5253,-0.5172,-0.4827,...,-0.0123,-0.0065,-0.1147,0.9944,0.7201,1.0752,0.6021,0.7217,1.1503,-0.3689
TCGA-05-4250-01,-1.0475,-0.8490,-0.7589,0.5110,0.6093,1.6944,0.7943,0.4315,0.5481,0.8785,...,0.7879,1.7046,0.8062,0.8387,-0.7846,0.4237,-0.5742,-0.3183,-2.1890,1.0850
TCGA-05-4382-01,-0.1971,0.4489,-0.0262,-0.3574,0.5930,0.5444,0.7155,0.4140,1.1524,0.8653,...,1.0030,0.4468,0.2823,0.5573,-0.2367,0.1216,-0.4074,-0.2508,0.1230,1.2968
TCGA-05-4384-01,0.0172,0.0055,0.2943,-0.8129,0.3294,-0.6369,-0.2955,-0.6180,-0.8997,-0.9871,...,-0.8031,-0.8720,0.4213,0.9711,-0.3327,0.3882,-0.4498,-1.4201,0.7751,-0.3105


In [ ]:
# Reset the index to turn the patient IDs into a column
gene_transposed = gene_transposed.reset_index()

In [ ]:
# Rename the 'index' column to 'patient_id' for clarity
gene_transposed = gene_transposed.rename(columns={"index": "patient_id"})

In [ ]:
# Extract the patient ID from the 'patient_id' column by taking the first 12 characters
gene_transposed["patient_id"] = gene_transposed["patient_id"].str[:12]

In [ ]:
gene_transposed.head()

Hugo_Symbol,patient_id,AKT1,ALK,ATM,BRAF,CCND1,CD274,CDKN2A,CTLA4,CXCL10,...,NRAS,PDCD1,PIK3CA,RB1,RET,ROS1,SMARCA4,STK11,TP53,VEGFA
0,TCGA-05-4244,-1.4348,0.1593,-1.8591,0.3711,1.2510,1.2876,-0.7482,-0.0031,-0.4304,...,0.5259,-1.0942,-0.2992,-0.4565,-1.1858,0.8186,-0.1573,0.4716,0.1371,2.1437
1,TCGA-05-4249,0.8508,0.2142,0.5106,-0.5496,-0.4023,0.0360,0.3018,-0.5253,-0.5172,...,-0.0123,-0.0065,-0.1147,0.9944,0.7201,1.0752,0.6021,0.7217,1.1503,-0.3689
2,TCGA-05-4250,-1.0475,-0.8490,-0.7589,0.5110,0.6093,1.6944,0.7943,0.4315,0.5481,...,0.7879,1.7046,0.8062,0.8387,-0.7846,0.4237,-0.5742,-0.3183,-2.1890,1.0850
3,TCGA-05-4382,-0.1971,0.4489,-0.0262,-0.3574,0.5930,0.5444,0.7155,0.4140,1.1524,...,1.0030,0.4468,0.2823,0.5573,-0.2367,0.1216,-0.4074,-0.2508,0.1230,1.2968
4,TCGA-05-4384,0.0172,0.0055,0.2943,-0.8129,0.3294,-0.6369,-0.2955,-0.6180,-0.8997,...,-0.8031,-0.8720,0.4213,0.9711,-0.3327,0.3882,-0.4498,-1.4201,0.7751,-0.3105


In [ ]:
# Save the cleaned gene expression data to a new CSV file
clinical_clean = pd.read_csv("LUAD_cleaned_data.csv")
clinical_clean.head()

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event,survival_5yr
0,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,Stage III,1,0.0
1,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,Stage III,1,0.0
2,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,Stage III,1,0.0
3,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,Stage II,1,0.0
4,TCGA-05-4402,8.021830,1:DECEASED,57.0,Female,Stage IV,1,0.0


In [ ]:
# Merge the cleaned clinical data with the transposed gene expression data on the 'patient_id' column
merged_data = pd.merge(
    clinical_clean,
    gene_transposed,
    on="patient_id",
    how="inner"
)

merged_data.shape

(212, 38)

In [ ]:
merged_data.columns

Index(['patient_id', 'overall_survival_months', 'overall_survival_status',
       'age', 'gender', 'stage', 'event', 'survival_5yr', 'AKT1', 'ALK', 'ATM',
       'BRAF', 'CCND1', 'CD274', 'CDKN2A', 'CTLA4', 'CXCL10', 'CXCL9', 'EGFR',
       'ERBB2', 'KEAP1', 'KRAS', 'MAPK1', 'MET', 'MKI67', 'MTOR', 'MYC', 'NF1',
       'NRAS', 'PDCD1', 'PIK3CA', 'RB1', 'RET', 'ROS1', 'SMARCA4', 'STK11',
       'TP53', 'VEGFA'],
      dtype='str')

In [ ]:
merged_data.isna().sum()

patient_id                 0
overall_survival_months    0
overall_survival_status    0
age                        8
gender                     0
stage                      0
event                      0
survival_5yr               0
AKT1                       0
ALK                        0
ATM                        0
BRAF                       0
CCND1                      0
CD274                      0
CDKN2A                     0
CTLA4                      0
CXCL10                     0
CXCL9                      0
EGFR                       0
ERBB2                      0
KEAP1                      0
KRAS                       0
MAPK1                      0
MET                        0
MKI67                      0
MTOR                       0
MYC                        0
NF1                        0
NRAS                       0
PDCD1                      0
PIK3CA                     0
RB1                        0
RET                        0
ROS1                       0
SMARCA4       

In [ ]:
merged_data.head()

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event,survival_5yr,AKT1,ALK,...,NRAS,PDCD1,PIK3CA,RB1,RET,ROS1,SMARCA4,STK11,TP53,VEGFA
0,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,Stage III,1,0.0,-1.0475,-0.8490,...,0.7879,1.7046,0.8062,0.8387,-0.7846,0.4237,-0.5742,-0.3183,-2.1890,1.0850
1,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,Stage III,1,0.0,-1.3604,-1.2188,...,1.4550,-0.1747,0.3443,-0.4809,-0.1480,-1.0888,-0.5807,0.5124,-0.0319,0.5732
2,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,Stage III,1,0.0,0.3895,-1.7834,...,2.6928,-1.0152,1.1727,1.4730,1.3009,0.6134,-0.1899,-3.5084,-2.3446,0.4621
3,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,Stage II,1,0.0,1.3291,3.0138,...,-0.0406,-0.6355,1.9261,-1.0100,0.4777,-0.6459,-0.3288,0.7419,-1.3342,-1.5368
4,TCGA-05-4402,8.021830,1:DECEASED,57.0,Female,Stage IV,1,0.0,1.3990,-0.6101,...,1.3335,-0.8319,0.6478,0.1487,-0.6852,0.4403,-0.1971,-0.2528,-2.0065,-0.2183


In [ ]:
merged_data.to_csv("luad_analytics_dataset.csv", index=False)

In [ ]:
test = pd.read_csv("luad_analytics_dataset.csv")
test.head()

,patient_id,overall_survival_months,overall_survival_status,age,gender,stage,event,survival_5yr,AKT1,ALK,...,NRAS,PDCD1,PIK3CA,RB1,RET,ROS1,SMARCA4,STK11,TP53,VEGFA
0,TCGA-05-4250,3.978039,1:DECEASED,79.0,Female,Stage III,1,0.0,-1.0475,-0.8490,...,0.7879,1.7046,0.8062,0.8387,-0.7846,0.4237,-0.5742,-0.3183,-2.1890,1.0850
1,TCGA-05-4395,0.000000,1:DECEASED,76.0,Male,Stage III,1,0.0,-1.3604,-1.2188,...,1.4550,-0.1747,0.3443,-0.4809,-0.1480,-1.0888,-0.5807,0.5124,-0.0319,0.5732
2,TCGA-05-4396,9.961535,1:DECEASED,76.0,Male,Stage III,1,0.0,0.3895,-1.7834,...,2.6928,-1.0152,1.1727,1.4730,1.3009,0.6134,-0.1899,-3.5084,-2.3446,0.4621
3,TCGA-05-4397,24.032613,1:DECEASED,65.0,Male,Stage II,1,0.0,1.3291,3.0138,...,-0.0406,-0.6355,1.9261,-1.0100,0.4777,-0.6459,-0.3288,0.7419,-1.3342,-1.5368
4,TCGA-05-4402,8.021830,1:DECEASED,57.0,Female,Stage IV,1,0.0,1.3990,-0.6101,...,1.3335,-0.8319,0.6478,0.1487,-0.6852,0.4403,-0.1971,-0.2528,-2.0065,-0.2183
